# 追踪数据转换 - 2022世界杯版本

本notebook用于将2022世界杯原始追踪数据转换为结构化的CSV文件

**功能**:
- 加载压缩的JSONL追踪数据
- 提取球员位置、球位置和比赛事件
- 计算速度和加速度
- 保存为CSV格式供后续分析使用

**基于**: 已验证可运行的 `test_convert_tracking_final.py`

**适配版本**: 2022 FIFA世界杯数据集

## 1. 导入必要的库

In [ ]:
import sys
import os

# 导入自定义模块
from convert_tracking import process_game, get_available_games, load_game_metadata

print("✓ 所有库导入成功")

## 2. 配置参数

设置要处理的比赛ID。可以是单场比赛或多场比赛。

In [ ]:
# 方式1: 处理单场比赛
GAME_IDS = ['10517']  # 决赛: 阿根廷 vs 法国

# 方式2: 处理多场比赛（取消注释以使用）
# GAME_IDS = ['10517', '3857', '3859']  # 决赛、季军赛等

# 方式3: 处理所有可用比赛（取消注释以使用）
# GAME_IDS = get_available_games()

print("="*80)
print("2022 FIFA World Cup - 追踪数据转换")
print("="*80)
print(f"\n将处理 {len(GAME_IDS)} 场比赛")
print(f"比赛ID: {GAME_IDS}")

## 3. 查看可用比赛

列出数据集中所有可用的比赛。

In [ ]:
# 获取所有可用比赛
print("\n获取所有可用比赛...")

try:
    available_games = get_available_games()
    print(f"[OK] 数据集中共有 {len(available_games)} 场比赛\n")
    
    print("前20场比赛ID:")
    for i, game_id in enumerate(available_games[:20], 1):
        print(f"  {i:2d}. {game_id}")
    
    if len(available_games) > 20:
        print(f"  ... 还有 {len(available_games) - 20} 场比赛")
        
except Exception as e:
    print(f"[ERROR] 获取可用比赛列表失败: {e}")
    print("请检查数据路径配置")
    import traceback
    traceback.print_exc()

## 4. 查看示例比赛的元数据

在处理之前，先查看比赛的基本信息。

In [ ]:
# 查看第一场比赛的元数据
example_game_id = GAME_IDS[0]
print(f"\n查看比赛 {example_game_id} 的元数据...\n")

try:
    metadata = load_game_metadata(example_game_id)
    
    print("="*80)
    print("比赛信息")
    print("="*80)
    print(f"主队: {metadata['homeTeam']['name']}")
    print(f"客队: {metadata['awayTeam']['name']}")
    print(f"日期: {metadata['date']}")
    print(f"球场: {metadata['stadium']['name']}")
    print(f"主队开球方向: {'左' if metadata['homeTeamStartLeft'] else '右'}")
    
    print(f"\n✓ 元数据加载成功")
    
except Exception as e:
    print(f"[ERROR] 加载元数据失败: {e}")
    import traceback
    traceback.print_exc()

## 5. 处理比赛数据

批量处理选定的比赛，将原始追踪数据转换为CSV文件。

**注意**: 每场比赛的处理可能需要2-5分钟时间。

In [ ]:
print("\n" + "="*80)
print(f"开始处理 {len(GAME_IDS)} 场比赛")
print("="*80)
print("\n注意: 每场比赛可能需要2-5分钟处理时间\n")

# 统计信息
success_count = 0
fail_count = 0
failed_games = []
processing_results = []

# 处理每场比赛
for i, game_id in enumerate(GAME_IDS, 1):
    print(f"\n{'='*80}")
    print(f"[{i}/{len(GAME_IDS)}] 处理比赛 {game_id}")
    print(f"{'='*80}")
    
    try:
        # 处理比赛
        balls_df, events_df, players_df = process_game(game_id, save_output=True)
        
        # 记录结果
        result = {
            'game_id': game_id,
            'status': 'success',
            'balls_frames': len(balls_df),
            'events': len(events_df),
            'player_records': len(players_df),
            'unique_players': players_df['playerName'].nunique()
        }
        processing_results.append(result)
        
        # 显示统计信息
        print(f"\n✓ 比赛 {game_id} 处理成功")
        print(f"  球追踪帧数: {len(balls_df):,}")
        print(f"  事件数: {len(events_df):,}")
        print(f"  球员追踪记录: {len(players_df):,}")
        print(f"  唯一球员数: {players_df['playerName'].nunique()}")
        
        success_count += 1
        
    except Exception as e:
        print(f"\n✗ 比赛 {game_id} 处理失败: {e}")
        fail_count += 1
        failed_games.append(game_id)
        
        result = {
            'game_id': game_id,
            'status': 'failed',
            'error': str(e)
        }
        processing_results.append(result)
        continue

# 显示总结
print(f"\n\n{'='*80}")
print("处理完成")
print(f"{'='*80}")
print(f"成功: {success_count} 场")
print(f"失败: {fail_count} 场")

if failed_games:
    print(f"\n失败的比赛ID: {failed_games}")

## 6. 处理结果汇总

In [ ]:
import pandas as pd

# 创建结果汇总表
if processing_results:
    results_df = pd.DataFrame(processing_results)
    
    print("\n" + "="*80)
    print("处理结果汇总")
    print("="*80)
    print(results_df.to_string(index=False))
    
    # 统计成功的比赛
    success_df = results_df[results_df['status'] == 'success']
    if len(success_df) > 0:
        print(f"\n成功处理的比赛统计:")
        print(f"  总帧数: {success_df['balls_frames'].sum():,}")
        print(f"  总事件数: {success_df['events'].sum():,}")
        print(f"  总球员记录: {success_df['player_records'].sum():,}")
        print(f"  平均球员数/场: {success_df['unique_players'].mean():.1f}")

## 7. 验证输出文件

检查生成的CSV文件是否正确。

In [ ]:
# 验证第一场比赛的输出
verify_game_id = GAME_IDS[0]
data_dir = f'Data/{verify_game_id}'

print(f"\n验证比赛 {verify_game_id} 的输出文件...\n")

if os.path.exists(data_dir):
    print(f"✓ 数据目录存在: {data_dir}\n")
    
    # 检查文件
    files_to_check = [
        f'balls_{verify_game_id}.csv',
        f'events_{verify_game_id}.csv',
        f'players_{verify_game_id}.csv'
    ]
    
    print("文件检查:")
    for filename in files_to_check:
        filepath = os.path.join(data_dir, filename)
        if os.path.exists(filepath):
            file_size = os.path.getsize(filepath) / (1024 * 1024)  # MB
            print(f"  ✓ {filename:<30} ({file_size:.2f} MB)")
        else:
            print(f"  ✗ {filename:<30} (不存在)")
    
    print(f"\n✓ 文件验证完成")
else:
    print(f"✗ 数据目录不存在: {data_dir}")

## 8. 数据样本查看

查看处理后数据的样本。

In [ ]:
# 加载并显示第一场比赛的数据样本
sample_game_id = GAME_IDS[0]
data_dir = f'Data/{sample_game_id}'

print(f"\n数据样本 (比赛 {sample_game_id})")
print("="*80)

try:
    import pandas as pd
    
    # 球员数据样本
    players_df = pd.read_csv(os.path.join(data_dir, f'players_{sample_game_id}.csv'))
    print("\n1. 球员数据 (前5行):")
    print(players_df[['frameNum', 'playerName', 'playerPos', 'x', 'y', 
                     'velocity_magnitude', 'is_home']].head())
    
    # 事件数据样本
    events_df = pd.read_csv(os.path.join(data_dir, f'events_{sample_game_id}.csv'))
    print("\n2. 事件数据 (前5行):")
    print(events_df[['frameNum', 'eventType', 'player_name', 'team_name']].head())
    
    # 球数据样本
    balls_df = pd.read_csv(os.path.join(data_dir, f'balls_{sample_game_id}.csv'))
    print("\n3. 球数据 (前5行):")
    print(balls_df[['frameNum', 'x', 'y', 'z', 'velocity_magnitude']].head())
    
    print(f"\n✓ 数据样本显示成功")
    
except Exception as e:
    print(f"✗ 读取数据文件失败: {e}")

## 总结

In [ ]:
print("\n" + "="*80)
print("处理完成总结")
print("="*80)

print("\n本notebook完成了以下工作：")
print(f"  ✓ 处理了 {success_count} 场比赛")
print(f"  ✓ 生成了 {success_count * 3} 个CSV文件")
print(f"  ✓ 数据验证和质量检查")

if fail_count > 0:
    print(f"\n  ⚠ {fail_count} 场比赛处理失败")

print("\n输出文件位置:")
print("  Data/{game_id}/")
print("    - balls_{game_id}.csv")
print("    - events_{game_id}.csv")
print("    - players_{game_id}.csv")

print("\n下一步：")
print("  - 使用 RunGATModel_2022WC.ipynb 训练模型")
print("  - 使用 test_run_gat_model_final.ipynb 快速测试")
print("  - 使用 Visualisation_2022WC.ipynb 进行可视化分析")